# 03 — Exploratory Data Analysis

This notebook explores the data to identify patterns, distributions, and relationships between variables.

## Objectives
- Understand distributions of key variables
- Compare engagement patterns across segments
- Identify correlations between engagement and renewal
- Visualize findings for the dashboard

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "processed")

# Load feature table
features_df = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features_vB.csv"))

print(f"Loaded {len(features_df)} customer records with features.")
features_df.head()

## 2. Distribution of Key Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Tenure distribution
axes[0, 0].hist(features_df['tenure'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Tenure Distribution')
axes[0, 0].set_xlabel('Tenure (months)')

# Monthly charges distribution
axes[0, 1].hist(features_df['monthly_charges'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Monthly Charges Distribution')
axes[0, 1].set_xlabel('Monthly Charges ($)')

# Total charges distribution
axes[0, 2].hist(features_df['total_charges'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 2].set_title('Total Charges Distribution')
axes[0, 2].set_xlabel('Total Charges ($)')

# Engagement score distribution
axes[1, 0].hist(features_df['engagement_score'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Engagement Score Distribution')
axes[1, 0].set_xlabel('Engagement Score')

# Total logins distribution
axes[1, 1].hist(features_df['total_logins'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Total Logins Distribution')
axes[1, 1].set_xlabel('Logins')

# Churn distribution
churn_counts = features_df['churn'].value_counts()
axes[1, 2].bar(churn_counts.index, churn_counts.values, edgecolor='black', alpha=0.7)
axes[1, 2].set_title('Churn Distribution')
axes[1, 2].set_xlabel('Churn Status')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../powerbi/screenshots/distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Engagement by Renewal Status

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Engagement score by churn
renewed = features_df[features_df['churn'] == 'No']['engagement_score']
churned = features_df[features_df['churn'] == 'Yes']['engagement_score']
axes[0, 0].hist([renewed, churned], bins=30, label=['Renewed', 'Churned'], alpha=0.7)
axes[0, 0].set_title('Engagement Score by Renewal Status')
axes[0, 0].set_xlabel('Engagement Score')
axes[0, 0].legend()

# Logins by churn
renewed_logins = features_df[features_df['churn'] == 'No']['total_logins']
churned_logins = features_df[features_df['churn'] == 'Yes']['total_logins']
axes[0, 1].hist([renewed_logins, churned_logins], bins=30, label=['Renewed', 'Churned'], alpha=0.7)
axes[0, 1].set_title('Total Logins by Renewal Status')
axes[0, 1].set_xlabel('Logins')
axes[0, 1].legend()

# Tenure by churn
renewed_tenure = features_df[features_df['churn'] == 'No']['tenure']
churned_tenure = features_df[features_df['churn'] == 'Yes']['tenure']
axes[1, 0].hist([renewed_tenure, churned_tenure], bins=30, label=['Renewed', 'Churned'], alpha=0.7)
axes[1, 0].set_title('Tenure by Renewal Status')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].legend()

# Monthly charges by churn
renewed_charges = features_df[features_df['churn'] == 'No']['monthly_charges']
churned_charges = features_df[features_df['churn'] == 'Yes']['monthly_charges']
axes[1, 1].hist([renewed_charges, churned_charges], bins=30, label=['Renewed', 'Churned'], alpha=0.7)
axes[1, 1].set_title('Monthly Charges by Renewal Status')
axes[1, 1].set_xlabel('Monthly Charges ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../powerbi/screenshots/engagement_by_renewal.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Renewal Rate by Segment

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By contract type
contract_renewal = features_df.groupby('contract').apply(
    lambda x: (x['churn'] == 'No').mean() * 100
).sort_values(ascending=False)
contract_renewal.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Renewal Rate by Contract Type')
axes[0, 0].set_ylabel('Renewal Rate (%)')
axes[0, 0].tick_params(axis='x', rotation=45)

# By tenure band
tenure_renewal = features_df.groupby('tenure_band').apply(
    lambda x: (x['churn'] == 'No').mean() * 100
).sort_values(ascending=False)
tenure_renewal.plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Renewal Rate by Tenure Band')
axes[0, 1].set_ylabel('Renewal Rate (%)')
axes[0, 1].tick_params(axis='x', rotation=45)

# By engagement band
engagement_renewal = features_df.groupby('engagement_band').apply(
    lambda x: (x['churn'] == 'No').mean() * 100
).sort_values(ascending=False)
engagement_renewal.plot(kind='bar', ax=axes[1, 0], color='seagreen')
axes[1, 0].set_title('Renewal Rate by Engagement Band')
axes[1, 0].set_ylabel('Renewal Rate (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

# By value segment
value_renewal = features_df.groupby('value_segment').apply(
    lambda x: (x['churn'] == 'No').mean() * 100
).sort_values(ascending=False)
value_renewal.plot(kind='bar', ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Renewal Rate by Value Segment')
axes[1, 1].set_ylabel('Renewal Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../powerbi/screenshots/renewal_by_segment.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = [
    'tenure', 'monthly_charges', 'total_charges',
    'engagement_score', 'total_interactions', 'total_content_views',
    'total_events', 'total_logins', 'total_sessions', 'total_features',
]
corr_matrix = features_df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix: Engagement & Subscription Metrics')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Key Observations

In [ ]:
# Renewal rate by engagement band
print("Renewal Rate by Engagement Band:")
engagement_renewal = features_df.groupby('engagement_band').apply(
    lambda x: pd.Series({
        'customers': len(x),
        'renewal_rate': (x['churn'] == 'No').mean() * 100,
        'avg_monthly_charges': x['monthly_charges'].mean(),
        'avg_tenure': x['tenure'].mean(),
    })
)
print(engagement_renewal.round(2))

print("\n\nRenewal Rate by Contract Type:")
contract_stats = features_df.groupby('contract').apply(
    lambda x: pd.Series({
        'customers': len(x),
        'renewal_rate': (x['churn'] == 'No').mean() * 100,
        'avg_monthly_charges': x['monthly_charges'].mean(),
        'avg_engagement': x['engagement_score'].mean(),
    })
)
print(contract_stats.round(2))